In [1]:
!pip install torch torchvision timm tqdm matplotlib


In [ ]:
from models.vit_dino import create_dino_and_define_train_mode
from data.datasets import get_cifar100, get_cifar100_transforms



In [ ]:
model = create_dino_and_define_train_mode()

In [ ]:
from data.datasets import get_cifar100, get_cifar100_transforms


In [ ]:
transform_train, transform_test = get_cifar100_transforms()

In [ ]:
train, val, test = get_cifar100(transform_train, transform_test, val_ratio=0.1, root="./data", seed=42)

In [ ]:
from data.partition import make_dataset_loaders

In [ ]:
train_loader, val_loader, test_loader = make_dataset_loaders(train, val, test)

In [ ]:
from data.partition import iid_shard, noniid_shard_by_nc_disjoint, make_client_loaders

In [ ]:
from fl.dataloaders import build_federated_dataloaders

In [16]:
client_loaders, val_loader, test_loader, *_ = build_federated_dataloaders(
    train_transform=transform_train,
    test_transform=transform_test,
    K=100,
    sharding="iid",
    val_ratio=0.1,
    batch_size=64,
    seed=42
)


In [17]:
client_loaders, val_loader, test_loader, *_ = build_federated_dataloaders(
    train_transform=transform_train,
    test_transform=transform_test,
    K=100,
    sharding="non_iid",
    Nc=1,
    val_ratio=0.1,
    batch_size=64,
    seed=42
)


In [18]:
print("N clientes:", len(client_loaders))
print("Tamanho cliente 0 (batches):", len(client_loaders[0]))
xb, yb = next(iter(client_loaders[0]))
print("Batch shape:", xb.shape, "Labels shape:", yb.shape)


N clientes: 100
Tamanho cliente 0 (batches): 8
Batch shape: torch.Size([64, 3, 224, 224]) Labels shape: torch.Size([64])


In [ ]:
from train.eval import evaluate

In [19]:
from utils.plots import plot_test_curves

In [ ]:
from train.trainer import run_training

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

runs = {}

for scheduler_name in ["cosine", "step", "multistep", "none"]:
    model = timm.create_model(
        'vit_small_patch16_224',
        pretrained=True,
        num_classes=100
    )

    hist, te_loss, te_acc = run_training(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        epochs=100,
        lr=0.03,
        wd=5e-4,
        scheduler_name=scheduler_name
    )

    runs[scheduler_name] = hist
    print(f"{scheduler_name}: final test acc = {te_acc:.4f}")


In [ ]:
plot_test_curves(runs, title_suffix="(ViT-S/16 DINO on CIFAR-100)")
